In [ ]:
import json
import re
from pathlib import Path
from collections import Counter

CVE_DIR = Path("../outputs/cves")
PATCH_DIR = Path("../outputs/patch")

_GITHUB_COMMIT_RE = re.compile(r"github\.com/[^/]+/[^/]+/commit/[0-9a-f]{5,}")
_GITHUB_PR_RE = re.compile(r"github\.com/[^/]+/[^/]+/pull/\d+")
_GITHUB_TAG_RE = re.compile(r"github\.com/[^/]+/[^/]+/releases/tag/[^/?#]+")

def classify_url(url: str) -> str:
    if _GITHUB_COMMIT_RE.search(url):
        return "github_commit"
    if _GITHUB_PR_RE.search(url):
        return "github_pr"
    if _GITHUB_TAG_RE.search(url):
        return "github_tag"
    if "github.com" in url:
        return "github_other"
    return "non_github"

cve_files = sorted(CVE_DIR.glob("*/*/CVE-*.json"))
patch_files = sorted(PATCH_DIR.glob("*.json"))

print(f"Total CVE files:   {len(cve_files)}")
print(f"Total patch files: {len(patch_files)}")

In [ ]:
cve_ids = {p.stem for p in cve_files}
patched_ids = {p.stem for p in patch_files}
total_cves = len(cve_ids)
with_patch = len(cve_ids & patched_ids)
without_patch = total_cves - with_patch

print(f"CVEs with patches:    {with_patch}  ({with_patch/total_cves:.1%})")
print(f"CVEs without patches: {without_patch}  ({without_patch/total_cves:.1%})")

In [ ]:
url_type_counter = Counter()
cves_with_github = set()
cves_with_non_github = set()
total_links = 0

for path in patch_files:
    data = json.loads(path.read_text())
    cve_id = data["cve_id"]
    for patch in data["patches"]:
        url = patch.get("url", "")
        kind = classify_url(url)
        url_type_counter[kind] += 1
        total_links += 1
        if kind.startswith("github"):
            cves_with_github.add(cve_id)
        else:
            cves_with_non_github.add(cve_id)

print(f"Total patch links: {total_links}")
print()
print("Link type breakdown:")
for kind, count in url_type_counter.most_common():
    print(f"  {kind}: {count}  ({count/total_links:.1%})")

print()
print(f"CVEs with ≥1 GitHub link:     {len(cves_with_github)}")
print(f"CVEs with ≥1 non-GitHub link: {len(cves_with_non_github)}")
print(f"CVEs with both:               {len(cves_with_github & cves_with_non_github)}")

In [ ]:
patch_counts = Counter()
for path in patch_files:
    data = json.loads(path.read_text())
    n = len(data["patches"])
    patch_counts[n] += 1

print("Patches per CVE:")
for n in sorted(patch_counts):
    print(f"  {n} patch(es): {patch_counts[n]} CVEs")

all_counts = [len(json.loads(p.read_text())["patches"]) for p in patch_files]
print(f"\nMin:  {min(all_counts)}")
print(f"Max:  {max(all_counts)}")
print(f"Mean: {sum(all_counts)/len(all_counts):.2f}")